# Practitioners Exploratory Data Analysis

## Purpose

Assess practitioner record completeness, identifier integrity, professional classifications, provider relationships, and downstream screening coverage.

## Files used

- `Practitioners.csv` — practitioner registration and verification details
- `Providers.csv` — provider records referenced by practitioners
- `Screenings.csv` — screenings delivered by practitioners

The source contains only three practitioners, so results describe the current records and should not be generalized to the wider practitioner population.

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
sns.set_theme(style="whitegrid")

def find_raw_data_dir(start: Path = Path.cwd()) -> Path:
    for directory in (start, *start.parents):
        for candidate in (directory / "data" / "raw", directory / "data-analytics" / "data" / "raw"):
            if candidate.is_dir():
                return candidate
    raise FileNotFoundError("Could not locate data-analytics/data/raw")

RAW_DATA_DIR = find_raw_data_dir()

## 1. Load the data

In [2]:
practitioners = pd.read_csv(RAW_DATA_DIR / "Practitioners.csv")
providers = pd.read_csv(RAW_DATA_DIR / "Providers.csv")
screenings = pd.read_csv(RAW_DATA_DIR / "Screenings.csv")

pd.DataFrame({
    "dataset": ["Practitioners", "Providers", "Screenings"],
    "rows": [len(practitioners), len(providers), len(screenings)],
    "columns": [len(practitioners.columns), len(providers.columns), len(screenings.columns)],
})

,dataset,rows,columns
0,Practitioners,3,8
1,Providers,2,6
2,Screenings,144,8


## 2. Inspect structure and completeness

In [3]:
practitioners

,practitioner_id,provider_id,practitioner_code,practitioner_type,registration_body,registration_number,verification_status,active
0,PRA-001,PRO-001,P80-PRA-001,nurse,BHPC,BHPC-RN-2814,verified,True
1,PRA-002,PRO-001,P80-PRA-002,nurse,BHPC,BHPC-RN-3176,verified,True
2,PRA-003,PRO-002,P80-PRA-003,fitness coach,other,FIT-BW-084,verified,True


In [4]:
practitioner_profile = pd.DataFrame({
    "data_type": practitioners.dtypes.astype(str),
    "missing_count": practitioners.isna().sum(),
    "missing_percent": practitioners.isna().mean().mul(100).round(1),
    "unique_values": practitioners.nunique(dropna=False),
})
practitioner_profile

,data_type,missing_count,missing_percent,unique_values
practitioner_id,object,0,0.0,3
provider_id,object,0,0.0,2
practitioner_code,object,0,0.0,3
practitioner_type,object,0,0.0,2
registration_body,object,0,0.0,2
registration_number,object,0,0.0,3
verification_status,object,0,0.0,1
active,bool,0,0.0,1


In [5]:
required_columns = [
    "practitioner_id", "provider_id", "practitioner_code",
    "practitioner_type", "registration_body",
    "registration_number", "verification_status", "active",
]
assert set(required_columns).issubset(practitioners.columns)
assert practitioners[required_columns].notna().all().all()
print("All required practitioner fields are present and complete.")

All required practitioner fields are present and complete.


## 3. Validate identifiers and registration records

In [6]:
identifier_quality = pd.Series({
    "missing_practitioner_ids": practitioners["practitioner_id"].isna().sum(),
    "duplicate_practitioner_ids": practitioners["practitioner_id"].duplicated().sum(),
    "invalid_practitioner_id_formats": (~practitioners["practitioner_id"].str.match(r"^PRA-[0-9]{3}$", na=False)).sum(),
    "duplicate_practitioner_codes": practitioners["practitioner_code"].duplicated().sum(),
    "invalid_practitioner_code_formats": (~practitioners["practitioner_code"].str.match(r"^P80-PRA-[0-9]{3}$", na=False)).sum(),
    "duplicate_registration_numbers": practitioners["registration_number"].duplicated().sum(),
})
identifier_quality.to_frame("count")

,count
missing_practitioner_ids,0
duplicate_practitioner_ids,0
invalid_practitioner_id_formats,0
duplicate_practitioner_codes,0
invalid_practitioner_code_formats,0
duplicate_registration_numbers,0


In [7]:
assert practitioners["practitioner_id"].is_unique
assert practitioners["practitioner_code"].is_unique
assert practitioners["registration_number"].is_unique
assert practitioners["practitioner_id"].str.match(r"^PRA-[0-9]{3}$", na=False).all()
assert practitioners["practitioner_code"].str.match(r"^P80-PRA-[0-9]{3}$", na=False).all()
print("Practitioner IDs, codes, and registration numbers are unique; internal identifiers are consistently formatted.")

Practitioner IDs, codes, and registration numbers are unique; internal identifiers are consistently formatted.


## 4. Review professional classifications and status

In [8]:
for column in ["practitioner_type", "registration_body", "verification_status", "active"]:
    display(practitioners[column].value_counts(dropna=False).to_frame("practitioner_count"))

,practitioner_count
practitioner_type,
nurse,2
fitness coach,1


,practitioner_count
registration_body,
BHPC,2
other,1


,practitioner_count
verification_status,
verified,3


,practitioner_count
active,
True,3


In [9]:
text_columns = [
    "practitioner_id", "provider_id", "practitioner_code",
    "practitioner_type", "registration_body",
    "registration_number", "verification_status",
]
whitespace_issues = pd.Series({
    column: practitioners[column].ne(practitioners[column].str.strip()).sum()
    for column in text_columns
})
whitespace_issues.to_frame("rows_with_outer_whitespace")

,rows_with_outer_whitespace
practitioner_id,0
provider_id,0
practitioner_code,0
practitioner_type,0
registration_body,0
registration_number,0
verification_status,0


In [ ]:
type_counts = practitioners["practitioner_type"].value_counts().sort_values()
ax = type_counts.plot(kind="barh", color="#176B87", figsize=(7, 3.5))
ax.set(title="Practitioners by type", xlabel="Practitioner count", ylabel="")
ax.bar_label(ax.containers[0])
plt.tight_layout()
plt.show()

## 5. Validate provider relationships

In [10]:
unknown_provider_ids = sorted(set(practitioners["provider_id"]) - set(providers["provider_id"]))
practitioner_provider_join = practitioners.merge(
    providers[["provider_id", "name", "provider_type", "verification_status", "active"]].rename(
        columns={"name": "provider_name", "verification_status": "provider_verification_status", "active": "provider_active"}
    ),
    on="provider_id", how="left", validate="many_to_one", indicator="provider_join_status",
)
print("Unknown provider IDs:", len(unknown_provider_ids))
practitioner_provider_join

Unknown provider IDs: 0


,practitioner_id,provider_id,practitioner_code,practitioner_type,registration_body,registration_number,verification_status,active,provider_name,provider_type,provider_verification_status,provider_active,provider_join_status
0,PRA-001,PRO-001,P80-PRA-001,nurse,BHPC,BHPC-RN-2814,verified,True,Bophelo Preventive Health Services,clinic,verified,True,both
1,PRA-002,PRO-001,P80-PRA-002,nurse,BHPC,BHPC-RN-3176,verified,True,Bophelo Preventive Health Services,clinic,verified,True,both
2,PRA-003,PRO-002,P80-PRA-003,fitness coach,other,FIT-BW-084,verified,True,ActiveWork Fitness Services,fitness company,verified,True,both


In [ ]:
assert not unknown_provider_ids
assert len(practitioner_provider_join) == len(practitioners)
assert practitioner_provider_join["provider_join_status"].eq("both").all()
print("All practitioners link to a known provider without row loss.")

## 6. Assess downstream screening coverage

In [11]:
unknown_screening_practitioners = sorted(set(screenings["practitioner_id"]) - set(practitioners["practitioner_id"]))
screenings_per_practitioner = (
    screenings.groupby("practitioner_id")["screening_id"].nunique().rename("screening_count").reset_index()
    .merge(practitioners[["practitioner_id", "practitioner_type", "provider_id"]], on="practitioner_id", how="right", validate="one_to_one")
    .fillna({"screening_count": 0})
    .sort_values("screening_count", ascending=False)
)
screenings_per_practitioner["screening_count"] = screenings_per_practitioner["screening_count"].astype(int)
print("Screenings with unknown practitioner IDs:", len(unknown_screening_practitioners))
screenings_per_practitioner

Screenings with unknown practitioner IDs: 0


,practitioner_id,screening_count,practitioner_type,provider_id
0,PRA-001,60,nurse,PRO-001
1,PRA-002,60,nurse,PRO-001
2,PRA-003,24,fitness coach,PRO-002


In [ ]:
assert not unknown_screening_practitioners
assert screenings["practitioner_id"].notna().all()
assert screenings_per_practitioner["screening_count"].sum() == screenings["screening_id"].nunique()
print("All screenings link to a known practitioner, and screening counts reconcile to the source.")

In [ ]:
ax = screenings_per_practitioner.sort_values("screening_count").plot(
    kind="barh", x="practitioner_id", y="screening_count", legend=False, color="#64CCC5", figsize=(7, 3.5)
)
ax.set(title="Recorded screenings by practitioner", xlabel="Unique screening count", ylabel="")
ax.bar_label(ax.containers[0])
plt.tight_layout()
plt.show()

## 7. Findings and recommendations

The current practitioner records are complete. Practitioner IDs, practitioner codes, and registration numbers are unique, while the internal identifiers follow their expected formats. All three practitioners are active and verified, and every practitioner links to a known provider. All 144 screening records also link to a known practitioner without row loss.

The current practitioner mix consists of two nurses and one fitness coach across two providers. Recorded screening volumes are 60, 60, and 24 respectively. These counts describe activity in the source data; they are not performance or productivity measures because service mix, assignment duration, working hours, and participant complexity are unavailable.

The dataset is too small and uniform to evaluate workforce diversity, verification risk, inactive practitioners, or meaningful workload patterns.

Next steps:

- Retain the current uniqueness, completeness, identifier-format, and relationship checks.
- Define controlled values for practitioner type, registration body, and verification status.
- Add registration expiry and verification dates to support compliance monitoring.
- Add practitioner capacity or working-hour fields before interpreting screening volume as workload.
- Repeat distribution and workload analysis when the practitioner population grows.